In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta 
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf 
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import pandas_ta as ta
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

In [12]:
class StockDataDownloader:
    @staticmethod    
    def download_stock_data(ticker, period="5y", progress=False):
        try:
            df = yf.download(ticker, period=period)
            print(f" {len(df)} trading days of data downloaded for {ticker} with period {period}.")
            print(f"  Date range: {df.index[0].date()} to {df.index[-1].date()}")
            return df
        except Exception as e:
            print(f"Error downloading data for {ticker}: {e}")
            return None
    @staticmethod
    def calculate_technical_indicators(df):
        df = df.copy()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        # Add technical indicators using pandas_ta    
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)
        df.ta.bbands(length=20, std=2, append=True)
        df.ta.stoch(length=14, append=True)
        df.ta.atr(length=14, append=True)
       # Calculate custom features
        df['Daily_Return'] = df['Close'].pct_change()
        df['ProcessLookupError'] = df['Close'].diff()
        df['Volume_Change'] = df['Volume'].pct_change()
        df['High_Low_Ratio'] = df['High'] / (df['Low']+ 1e-8)
        df['Close_Open_Ratio'] = df['Close'] / (df['Open'] + 1e-8)
        
        #Rolling statistics
        df['Price_MA10'] = df['Close'].rolling(window=10).mean()
        df['Price_MA20'] = df['Close'].rolling(window=20).mean()
        df['Volume_MA'] = df['Volume'].rolling(window=20).mean()
        return df.dropna()
    def create_target_variable(df, shift_days=1):
        df = df.copy()
        
        next_return = df['Daily_Return'].shift(-shift_days)
        target = np.zeros(len(df))
        target[next_return < -0.01] = 0 # Down
        target[(next_return >= -0.01) & (next_return <= 0.01)] = 1 # Neutral
        target[next_return > 0.01] = 2 # Up
        df['Target'] = target 
        return df
    
class FeatureEngineer:
    @staticmethod
    def select_features(df):
        exclude_cols = ['Target', 'Dividens', 'Stock Splits']
        features = [col for col in df.columns if col not in exclude_cols and 
                    df[col].dtype in ['float64', 'float32', 'int64', 'int32']]
        if len(features) < 10:
            if 'Close' in df.columns:
                return df[['Open', 'High', 'Low', 'Close', 'Volume']].values
        return df[features].values
    
    @staticmethod
    def normalize_features(X_train, X_val, X_test):
        scaler = MinMaxScaler()
        X_trained_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        return X_trained_scaled, X_val_scaled, X_test_scaled, scaler
    
    @staticmethod
    def create_sequences(X, y, seq_length=20, stride=1):
        """
        Create sequences for time series models
        
        Args:
            X: Feature matrix
            y: Target vector
            seq_length: Sequence length (days)
            stride: Stride between sequences
        
        Returns:
            X_seq: (n_samples, seq_length, n_features)
            y_seq: Target values
        """
        
        X_seq, y_seq = [], []
        
        for i in range(0, len(X) - seq_length, stride):
            X_seq.append(X[i:i + seq_length])
            y_seq.append(y[i + seq_length - 1])  # Target at end of sequence
        
        return np.array(X_seq), np.array(y_seq)
    
    
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
    
        

In [13]:
def calculate_rsi(prices, period=14):
    """Calculate RSI manually"""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))
    return rsi
def calculate_macd(prices, fast=12, slow=26):
    """Calculate MACD manually"""
    ema_fast = prices.ewm(span=fast).mean()
    ema_slow = prices.ewm(span=slow).mean()
    macd = ema_fast - ema_slow
    return macd
def calculate_bollinger_width(prices, window=20, num_std=2):
    """Calculate Bollinger Band width"""
    ma = prices.rolling(window).mean()
    std = prices.rolling(window).std()
    bb_width = (2 * num_std * std) / ma
    return bb_width

In [14]:
class CNNClassifier(nn.Module):
    def __init__(self, n_features=16, seq_length=20, n_classes=3):
        super(CNNClassifier, self).__init__()
        
        # Encoder: Extract features
        self.conv1 = nn.Conv1d(n_features, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        
        # Calculate size after convolutions
        size_after_conv = seq_length // 8 * 128
        
        # Classifier
        self.fc1 = nn.Linear(size_after_conv, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, n_classes)
    
    def forward(self, x):
        # (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.transpose(1, 2)
        
        # Convolutional layers
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = self.dropout(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Classification
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        
        return x
    
class TDNNClassifier(nn.Module):
    """TDNN for stock direction classification"""
    
    def __init__(self, n_features=16, seq_length=20, n_classes=3, time_delays=(1, 2, 4, 8)):
        super(TDNNClassifier, self).__init__()
        
        self.seq_length = seq_length
        self.time_delays = time_delays
        
        # Number of delayed features
        n_delayed = 1 + len(time_delays)
        
        # Dense layers
        self.fc1 = nn.Linear(seq_length * n_features * n_delayed, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, n_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, features)
        batch_size = x.size(0)
        n_features = x.size(2)
        
        # Create time-delayed versions
        delayed_features = [x]
        
        for delay in self.time_delays:
            if delay > 0:
                # Pad and shift
                delayed = torch.cat([
                    torch.zeros(batch_size, delay, n_features, device=x.device),
                    x[:, :-delay, :]
                ], dim=1)
                delayed_features.append(delayed)
        
        # Concatenate all
        x_delayed = torch.cat(delayed_features, dim=2)  # (batch, seq_len, features*(1+delays))
        
        # Flatten
        x = x_delayed.view(batch_size, -1)
        
        # Dense layers
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        
        return x

In [15]:
def train_classifier(model, train_loader, val_loader, epochs=50, model_name='Model'):
    print(f'\n{'='*80}')
    print(f'Training {model_name}')
    print(f'{'='*80}\n')
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() 
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else: 
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}')
    return train_losses, val_losses

def evaluate_classifier(model, test_loader, model_name="Model"):
    """Evaluate the model on the test set and print metrics"""
    print(f"\n{'='*60}")
    print(f" Evaluating {model_name} on Test Set ")
    print(f"{'='*60}\n")
    model.eval()
    
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)
    y_probs = np.concatenate(all_probs) 
        
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    
    print(f"\nClassification Report:")
    class_names = ['Down', 'Neutral', 'Up']
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'predictions': y_pred,
        'targets': y_true,
        'probabilities': y_probs
    }

In [16]:
def main():
    
    print("\n" + "="*80)
    print("PHASE 1: DOWNLOAD REAL STOCK DATA")
    print("="*80)
    
    # Download multiple stocks
    tickers = ['^GSPC', 'AAPL', 'MSFT', 'AMZN']  # S&P 500, Apple, Microsoft, Amazon
    
    all_data = {}
    for ticker in tickers:
        df = StockDataDownloader.download_stock_data(ticker, period="5y")
        if df is not None:
            df = StockDataDownloader.calculate_technical_indicators(df)
            df = StockDataDownloader.create_target_variable(df)
            all_data[ticker] = df
            print(f"   Features calculated: {df.shape[1]} columns, {df.shape[0]} rows")
    
    # Use S&P 500 as primary
    primary_ticker = '^GSPC'
    if primary_ticker not in all_data:
        print("✗ Could not download data. Using synthetic data for demonstration...")
        # Fallback to synthetic
        all_data = create_synthetic_data()
    
    # ========== PHASE 2: FEATURE ENGINEERING ==========
    
    print("\n" + "="*80)
    print("PHASE 2: FEATURE ENGINEERING")
    print("="*80)
    
    ticker = 'AAPL'
    df = all_data[ticker]
    
    print(f"\nSelected ticker: {ticker}")
    print(f"Data shape: {df.shape}")
    
    # Select features
    X = FeatureEngineer.select_features(df)
    y = df['Target'].values
    
    print(f"Features shape: {X.shape}")
    print(f"Classes distribution: Down={sum(y==0)}, Neutral={sum(y==1)}, Up={sum(y==2)}")
    
    # Create sequences
    seq_length = 20  # 20 trading days
    X_seq, y_seq = FeatureEngineer.create_sequences(X, y, seq_length=seq_length, stride=1)
    
    print(f"Sequences shape: {X_seq.shape}")
    
    # Train/val/test split
    X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    
    # Normalize
    X_train_norm, X_val_norm, X_test_norm, scaler = FeatureEngineer.normalize_features(
        X_train.reshape(-1, X_train.shape[-1]),
        X_val.reshape(-1, X_val.shape[-1]),
        X_test.reshape(-1, X_test.shape[-1])
    )
    
    X_train_norm = X_train_norm.reshape(X_train.shape)
    X_val_norm = X_val_norm.reshape(X_val.shape)
    X_test_norm = X_test_norm.reshape(X_test.shape)
    
    print(f"\nTrain: {X_train_norm.shape[0]}, Val: {X_val_norm.shape[0]}, Test: {X_test_norm.shape[0]}")
    
    # ========== PHASE 3: CREATE DATALOADERS ==========
    
    print("\n" + "="*80)
    print("PHASE 3: CREATE DATALOADERS")
    print("="*80)
    
    batch_size = 32
    train_ds = StockDataset(X_train_norm, y_train)
    val_ds = StockDataset(X_val_norm, y_val)
    test_ds = StockDataset(X_test_norm, y_test)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    
    print(f"✓ DataLoaders created (batch_size={batch_size})")
    
    # ========== PHASE 4: TRAIN MODELS ==========
    
    print("\n" + "="*80)
    print("PHASE 4: TRAIN MODELS")
    print("="*80)
    
    n_features = X_train_norm.shape[2]
    
    # CNN
    print("\n[4.1] CNN Classifier")
    model_cnn = CNNClassifier(n_features=n_features, seq_length=seq_length, n_classes=3)
    train_classifier(model_cnn, train_loader, val_loader, epochs=50, model_name="CNN Classifier")
    
    # TDNN
    print("\n[4.2] TDNN Classifier")
    model_tdnn = TDNNClassifier(n_features=n_features, seq_length=seq_length, n_classes=3)
    train_classifier(model_tdnn, train_loader, val_loader, epochs=50, model_name="TDNN Classifier")
    
    # ========== PHASE 5: EVALUATE ==========
    
    print("\n" + "="*80)
    print("PHASE 5: EVALUATE MODELS")
    print("="*80)
    
    results_cnn = evaluate_classifier(model_cnn, test_loader, "CNN Classifier")
    results_tdnn = evaluate_classifier(model_tdnn, test_loader, "TDNN Classifier")
    
    # ========== PHASE 6: RESULTS SUMMARY ==========
    
    print("\n" + "="*80)
    print("PHASE 6: RESULTS SUMMARY")
    print("="*80)
    
    print("\nCNN Classifier:")
    print(f"  Accuracy: {results_cnn['accuracy']:.4f}")
    print(f"  F1-Score: {results_cnn['f1']:.4f}")
    
    print("\nTDNN Classifier:")
    print(f"  Accuracy: {results_tdnn['accuracy']:.4f}")
    print(f"  F1-Score: {results_tdnn['f1']:.4f}")
    
    best_model = "CNN" if results_cnn['f1'] > results_tdnn['f1'] else "TDNN"
    print(f"\n➜ Best Model: {best_model}")
    
    # Save results
    np.savez('stock_prediction_results.npz',
             predictions_cnn=results_cnn['predictions'],
             predictions_tdnn=results_tdnn['predictions'],
             targets=y_test,
             probabilities_cnn=results_cnn['probabilities'],
             probabilities_tdnn=results_tdnn['probabilities'])
    
    print("\n✓ Results saved to stock_prediction_results.npz")
    
    return {
        'models': {'cnn': model_cnn, 'tdnn': model_tdnn},
        'results': {'cnn': results_cnn, 'tdnn': results_tdnn},
        'data': {'X_test': X_test_norm, 'y_test': y_test},
        'scaler': scaler
    }
 
 
def create_synthetic_data():
    """Create realistic synthetic stock data as fallback"""
    
    print("Creating realistic synthetic stock data...")
    
    np.random.seed(42)
    n_days = 1250
    dates = pd.date_range(end=datetime.now(), periods=n_days, freq='D')
    
    # Generate realistic OHLCV data using GBM
    returns = np.random.normal(0.0005, 0.015, n_days)
    close = 100 * np.exp(np.cumsum(returns))
    
    # High and Low
    high = close + np.abs(np.random.normal(0, 0.8, n_days))
    low = close - np.abs(np.random.normal(0, 0.8, n_days))
    
    # Open and Close with correlation
    open_price = np.concatenate([[100], close[:-1] + np.random.normal(0, 0.3, n_days-1)])
    
    # Volume with correlation to price movement
    volume_base = np.random.uniform(5e6, 50e6, n_days)
    volume = volume_base * (1 + np.abs(returns) * 2)
    
    df = pd.DataFrame({
        'Open': open_price,
        'High': high,
        'Low': low,
        'Close': close,
        'Volume': volume
    }, index=dates)
    
    # Add features
    df['Daily_Return'] = df['Close'].pct_change()
    df['Price_Change'] = df['Close'].diff()
    df['Volume_Change'] = df['Volume'].pct_change()
    df['High_Low_Ratio'] = df['High'] / (df['Low'] + 1e-8)
    df['Close_Open_Ratio'] = df['Close'] / (df['Open'] + 1e-8)
    
    # Technical indicators (manual calculation)
    df['RSI'] = calculate_rsi(df['Close'], 14)
    df['MACD'] = calculate_macd(df['Close'])
    df['BB_Width'] = calculate_bollinger_width(df['Close'], 20, 2)
    df['Price_MA10'] = df['Close'].rolling(10).mean()
    df['Price_MA20'] = df['Close'].rolling(20).mean()
    df['Volume_MA'] = df['Volume'].rolling(20).mean()
    df['Volatility'] = df['Daily_Return'].rolling(20).std()
    
    # Create target
    df['Next_Return'] = df['Daily_Return'].shift(-1)
    df['Target'] = 1  # Neutral
    df.loc[df['Next_Return'] < -0.01, 'Target'] = 0  # Down
    df.loc[df['Next_Return'] > 0.01, 'Target'] = 2  # Up
    
    return {'^GSPC': df.dropna()}

In [ ]:
if __name__ == "__main__":
    results = main()
    print("\n" + "="*80)
    print("PROJECT COMPLETE")
    print("="*80)


PHASE 1: DOWNLOAD REAL STOCK DATA


[*********************100%***********************]  1 of 1 completed


 1255 trading days of data downloaded for ^GSPC with period 5y.
  Date range: 2021-05-28 to 2026-05-28
   Features calculated: 27 columns, 1222 rows


[*********************100%***********************]  1 of 1 completed


 1255 trading days of data downloaded for AAPL with period 5y.
  Date range: 2021-05-28 to 2026-05-28
   Features calculated: 27 columns, 1222 rows


[*********************100%***********************]  1 of 1 completed


 1255 trading days of data downloaded for MSFT with period 5y.
  Date range: 2021-05-28 to 2026-05-28
   Features calculated: 27 columns, 1222 rows


[*********************100%***********************]  1 of 1 completed


 1255 trading days of data downloaded for AMZN with period 5y.
  Date range: 2021-05-28 to 2026-05-28
   Features calculated: 27 columns, 1222 rows

PHASE 2: FEATURE ENGINEERING

Selected ticker: AAPL
Data shape: (1222, 27)
Features shape: (1222, 26)
Classes distribution: Down=251, Neutral=676, Up=295
Sequences shape: (1202, 20, 26)

Train: 768, Val: 193, Test: 241

PHASE 3: CREATE DATALOADERS
✓ DataLoaders created (batch_size=32)

PHASE 4: TRAIN MODELS

[4.1] CNN Classifier

Training CNN Classifier

Epoch 1/50 - Train Loss: 1.0321 - Val Loss: 1.0485
Epoch 10/50 - Train Loss: 0.9767 - Val Loss: 1.0333
Early stopping at epoch 15

[4.2] TDNN Classifier

Training TDNN Classifier

Epoch 1/50 - Train Loss: 1.0665 - Val Loss: 1.0357
Epoch 10/50 - Train Loss: 0.9492 - Val Loss: 1.0077
Epoch 20/50 - Train Loss: 0.8821 - Val Loss: 0.9791
Early stopping at epoch 30

PHASE 5: EVALUATE MODELS

 Evaluating CNN Classifier on Test Set 

Accuracy:  0.5519
Precision: 0.5919
Recall:    0.5519
F1-Score: 

: 